In [57]:
import numpy as np
import pandas as pd
import os
from scipy.stats import beta
from scipy.special import digamma, polygamma
from sklearn.metrics import mean_squared_error
from GASModels.distributions.gb2_log_link import GB2LogLinkDistribution
from GASModels.distributions.za_gb2_log_link import ZAGB2LogLinkDistribution

In [65]:
INPUT = r"C:\\Users\\ilang\\OneDrive\\Documentos\\Ilan\\academia\\dissertação\\data\\inmet\\inmet 250126\\treated"
os.listdir(INPUT)[0]

df = pd.read_csv(os.path.join(INPUT, os.listdir(INPUT)[1]), sep=";")

In [8]:
df['precipitation sample'] = df['precipitation original'].copy()
sample = df.sample(n=200)
df['precipitation sample'].iloc[sample.index] = np.nan

C:\Users\ilang\AppData\Local\Temp\ipykernel_37440\4107092145.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['precipitation sample'].iloc[sample.index] = np.nan
C:\Users\ilang\AppData\Local\Temp\ipykernel_37440\4107092145.py:3: Settin

In [18]:
df['fill interpolation'] = df['precipitation sample'].astype(float).interpolate(method = 'linear').values

df['doy'] = pd.to_datetime(df['Data Medicao'], format="%d/%m/%Y").dt.dayofyear
clim = df.groupby('doy')['precipitation original'].mean()
df['fill doy'] = df['precipitation original'].fillna(df['doy'].map(clim)).values

df['fill smoothed original'] = df['precipitation sample'].fillna(df['smoothed'].str.replace(",", ".").astype(float)).astype(float).values 

df['fill smoothed log'] = df['precipitation sample'].fillna(df['smoothed log'].str.replace(",", ".").astype(float).apply(lambda x: np.exp(x) - 0.01)).astype(float).values 

In [26]:
df['precipitation original'].dropna().values

array([63.9,  0. ,  0.4, ...,  5.8,  0. ,  0. ], shape=(5441,))

In [28]:
len(df)

5508

In [34]:
df[df['precipitation sample'].isna()]

,Unnamed: 0,Data Medicao,"PRECIPITACAO TOTAL, DIARIO(mm)",Unnamed: 2,year,precipitation original,precipitation log,location,smoothed,smoothed log,precipitation sample,fill interpolation,doy,fill doy,fill smoothed original,fill smoothed log
45,45,15/02/2000,21,NaN,2000,21.0,3.044.998.514.856.900,CRUZEIRO DO SUL (ACRE),"8,720751681","0,117997186",NaN,17.00,46,21.000000,8.720752,1.115241
49,49,19/02/2000,"4,6",NaN,2000,4.6,1.528.227.857.008.550,CRUZEIRO DO SUL (ACRE),"8,645960157","0,248279392",NaN,3.65,50,4.600000,8.645960,1.271818
62,62,03/03/2000,0,NaN,2000,0.0,-4.605.170.185.988.090,CRUZEIRO DO SUL (ACRE),"8,794223629","0,034232596",NaN,7.45,63,0.000000,8.794224,1.024825
66,66,07/03/2000,NaN,NaN,2000,NaN,NaN,CRUZEIRO DO SUL (ACRE),"9,105988249","-0,10595824",NaN,9.35,67,7.157143,9.105988,0.889462
109,109,19/04/2000,0,NaN,2000,0.0,-4.605.170.185.988.090,CRUZEIRO DO SUL (ACRE),"8,005498879","-0,400249657",NaN,0.30,110,0.000000,8.005499,0.660153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5363,5363,07/09/2014,NaN,NaN,2014,NaN,NaN,CRUZEIRO DO SUL (ACRE),"4,160298081","-2,511272901",NaN,0.00,250,4.630769,4.160298,0.071165
5377,5377,21/09/2014,0,NaN,2014,0.0,-4.605.170.185.988.090,CRUZEIRO DO SUL (ACRE),"5,552879319","-2,303144461",NaN,25.45,264,0.000000,5.552879,0.089944
5413,5413,27/10/2014,NaN,NaN,2014,NaN,NaN,CRUZEIRO DO SUL (ACRE),"6,64014539","-1,210984937",NaN,0.10,300,6.607143,6.640145,0.287904
5422,5422,05/11/2014,0,NaN,2014,0.0,-4.605.170.185.988.090,CRUZEIRO DO SUL (ACRE),"7,185009993","-1,282394966",NaN,17.50,309,0.000000,7.185010,0.267372


In [30]:
5508 - 5441

67

In [27]:
5441 - 5243

198

In [35]:
temp = df[['precipitation original', 'fill interpolation']]
mse = mean_squared_error(temp.dropna()['precipitation original'].values, temp.dropna()['fill interpolation'].values)

In [37]:
np.sqrt(mse)

np.float64(2.76254690283242)

In [58]:
REQUIRED_COLS = {
    "precipitation original",
    "Data Medicao",
    "smoothed",
    "smoothed log"
}

valid_files = []

for fname in os.listdir(INPUT):
    try:
        df_test = pd.read_csv(os.path.join(INPUT, fname), sep=";", nrows=5)
        if REQUIRED_COLS.issubset(df_test.columns):
            valid_files.append(fname)
    except Exception:
        pass

print(f"found {len(valid_files)} valid files")


found 3 valid files


In [67]:
import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

# ======================================================
# CONFIG
# ======================================================
INPUT = r"C:\\Users\\ilang\\OneDrive\\Documentos\\Ilan\\academia\\dissertação\\data\\inmet\\inmet 250126\\treated"

N_MASK = 2000
MIN_REP = 30
MAX_REP = 1000

TOL = 0.03        # 3% margin of error
Z = 1.96          # 95% CI

METHODS = [
    "interpolation",
    "doy",
    "smoothed_original",
    "smoothed_log"
]

REQUIRED_COLS = {
    "precipitation original",
    "Data Medicao",
    "smoothed",
    "smoothed log"
}


# ======================================================
# Find valid files (robust, not index-based)
# ======================================================
valid_files = []

for fname in os.listdir(INPUT):
    try:
        tmp = pd.read_csv(os.path.join(INPUT, fname), sep=";", nrows=3)
        if REQUIRED_COLS.issubset(tmp.columns):
            valid_files.append(fname)
    except:
        pass

print("Valid files:", valid_files)


# ======================================================
# MONTE CARLO
# ======================================================
results = []

for file_idx, filename in enumerate(valid_files):

    print(f"\nProcessing {filename}")

    # --------------------------
    # Load once
    # --------------------------
    df0 = pd.read_csv(os.path.join(INPUT, filename), sep=";")

    orig = df0["precipitation original"].astype(float).to_numpy()

    doy = pd.to_datetime(
        df0["Data Medicao"], format="%d/%m/%Y"
    ).dt.dayofyear

    clim = df0.groupby(doy)["precipitation original"].mean()
    doy_map = doy.map(clim).fillna(0).to_numpy()

    smoothed = df0["smoothed"].str.replace(",", ".").astype(float).to_numpy()

    smoothed_log = (
        df0["smoothed log"]
        .str.replace(",", ".")
        .astype(float)
        .apply(lambda x: np.exp(x) - 0.01)
        .to_numpy()
    )

    valid_idx = np.where(~np.isnan(orig))[0]

    mse_store = {m: [] for m in METHODS}
    rep = 0

    # ==================================================
    # FAST ADAPTIVE MC LOOP (NUMPY ONLY)
    # ==================================================
    while True:

        rep += 1

        sample = orig.copy()

        mask_idx = np.random.choice(valid_idx, size=N_MASK, replace=False)
        sample[mask_idx] = np.nan

        # ------------- imputations -------------

        # interpolation (only place we use pandas)
        interp = (
            pd.Series(sample)
            .interpolate(method="linear", limit_direction="both")
            .to_numpy()
        )

        doy_fill = np.where(np.isnan(sample), doy_map, sample)
        smooth_fill = np.where(np.isnan(sample), smoothed, sample)
        smooth_log_fill = np.where(np.isnan(sample), smoothed_log, sample)

        fills = {
            "interpolation": interp,
            "doy": doy_fill,
            "smoothed_original": smooth_fill,
            "smoothed_log": smooth_log_fill
        }

        y_true = orig[mask_idx]

        for method, y_hat in fills.items():

            y_pred = y_hat[mask_idx]

            if np.isnan(y_pred).any():
                continue

            mse = np.mean((y_true - y_pred) ** 2)
            mse_store[method].append(mse)

        # ------------- CI stopping -------------
        if rep >= MIN_REP:

            done = True

            for vals in mse_store.values():
                mu = np.mean(vals)
                sd = np.std(vals, ddof=1)

                half_ci = Z * sd / np.sqrt(len(vals))

                if half_ci > TOL * mu:
                    done = False
                    break

            if done or rep >= MAX_REP:
                break

    print(f"Stopped at R = {rep}")

    # store results
    for method, vals in mse_store.items():
        for r, mse in enumerate(vals):
            results.append({
                "file": filename,
                "file_idx": file_idx,
                "rep": r,
                "method": method,
                "mse": mse
            })


# ======================================================
# RAW RESULTS
# ======================================================
mse_df = pd.DataFrame(results)


# ======================================================
# SUMMARY TABLE (mean ± CI)
# ======================================================
summary_df = (
    mse_df
    .groupby(["file", "method"])
    .agg(
        mean=("mse", "mean"),
        std=("mse", "std"),
        n=("mse", "count")
    )
    .reset_index()
)

summary_df["half_ci"] = Z * summary_df["std"] / np.sqrt(summary_df["n"])
summary_df["rel_error_%"] = 100 * summary_df["half_ci"] / summary_df["mean"]

print("\n=== MSE SUMMARY ===")
print(summary_df)


# ======================================================
# PAIRED MEANS TESTS
# ======================================================
tests = []

for file in mse_df["file"].unique():

    sub = mse_df[mse_df["file"] == file]
    wide = sub.pivot(index="rep", columns="method", values="mse")

    methods = wide.columns

    for i in range(len(methods)):
        for j in range(i+1, len(methods)):

            m1, m2 = methods[i], methods[j]

            tstat, pval = ttest_rel(wide[m1], wide[m2])

            tests.append({
                "file": file,
                "method_A": m1,
                "method_B": m2,
                "mean_A": wide[m1].mean(),
                "mean_B": wide[m2].mean(),
                "diff(A-B)": wide[m1].mean() - wide[m2].mean(),
                "t_stat": tstat,
                "p_value": pval
            })

tests_df = pd.DataFrame(tests)

print("\n=== PAIRED MEANS TESTS ===")
print(tests_df)


Valid files: ['BELO HORIZONTE.csv', 'CRUZEIRO DO SUL (ACRE).csv', 'GARANHUNS (PERNAMBUCO).csv']

Processing BELO HORIZONTE.csv
Stopped at R = 39

Processing CRUZEIRO DO SUL (ACRE).csv
Stopped at R = 30

Processing GARANHUNS (PERNAMBUCO).csv
Stopped at R = 90

=== MSE SUMMARY ===
                          file             method        mean        std   n  \
0           BELO HORIZONTE.csv                doy  121.979653  11.039301  39   
1           BELO HORIZONTE.csv      interpolation  141.814624  10.277362  39   
2           BELO HORIZONTE.csv       smoothed_log  145.437270  13.815523  39   
3           BELO HORIZONTE.csv  smoothed_original  115.949711  10.551021  39   
4   CRUZEIRO DO SUL (ACRE).csv                doy  139.857144   9.936142  30   
5   CRUZEIRO DO SUL (ACRE).csv      interpolation  227.384554  11.897439  30   
6   CRUZEIRO DO SUL (ACRE).csv       smoothed_log  184.924756  13.084903  30   
7   CRUZEIRO DO SUL (ACRE).csv  smoothed_original  148.119053  11.181875  30   


In [70]:
summary_df

,file,method,mean,std,n,half_ci,rel_error_%
0,BELO HORIZONTE.csv,doy,121.979653,11.039301,39,3.464698,2.840390
1,BELO HORIZONTE.csv,interpolation,141.814624,10.277362,39,3.225562,2.274492
2,BELO HORIZONTE.csv,smoothed_log,145.437270,13.815523,39,4.336018,2.981367
3,BELO HORIZONTE.csv,smoothed_original,115.949711,10.551021,39,3.311450,2.855937
4,CRUZEIRO DO SUL (ACRE).csv,doy,139.857144,9.936142,30,3.555603,2.542310
5,CRUZEIRO DO SUL (ACRE).csv,interpolation,227.384554,11.897439,30,4.257444,1.872354
6,CRUZEIRO DO SUL (ACRE).csv,smoothed_log,184.924756,13.084903,30,4.682373,2.532042
7,CRUZEIRO DO SUL (ACRE).csv,smoothed_original,148.119053,11.181875,30,4.001382,2.701464
8,GARANHUNS (PERNAMBUCO).csv,doy,45.706807,6.313366,90,1.304355,2.853743
9,GARANHUNS (PERNAMBUCO).csv,interpolation,51.163874,6.416176,90,1.325596,2.590882


In [71]:
def highlight_winner(row):
    return [
        "background-color: lightgreen" if row["diff(A-B)"] < 0 else ""
    ] * len(row)

styled_tests = tests_df.style.apply(highlight_winner, axis=1)

styled_tests



,file,method_A,method_B,mean_A,mean_B,diff(A-B),t_stat,p_value
0,BELO HORIZONTE.csv,doy,interpolation,121.979653,141.814624,-19.834971,-14.675795,0.000000
1,BELO HORIZONTE.csv,doy,smoothed_log,121.979653,145.437270,-23.457618,-36.217403,0.000000
2,BELO HORIZONTE.csv,doy,smoothed_original,121.979653,115.949711,6.029942,21.032000,0.000000
3,BELO HORIZONTE.csv,interpolation,smoothed_log,141.814624,145.437270,-3.622646,-2.253246,0.030096
4,BELO HORIZONTE.csv,interpolation,smoothed_original,141.814624,115.949711,25.864913,19.817514,0.000000
5,BELO HORIZONTE.csv,smoothed_log,smoothed_original,145.437270,115.949711,29.487560,50.019244,0.000000
6,CRUZEIRO DO SUL (ACRE).csv,doy,interpolation,139.857144,227.384554,-87.527411,-48.351214,0.000000
7,CRUZEIRO DO SUL (ACRE).csv,doy,smoothed_log,139.857144,184.924756,-45.067612,-64.076871,0.000000
8,CRUZEIRO DO SUL (ACRE).csv,doy,smoothed_original,139.857144,148.119053,-8.261909,-21.875046,0.000000
9,CRUZEIRO DO SUL (ACRE).csv,interpolation,smoothed_log,227.384554,184.924756,42.459799,19.620442,0.000000


In [73]:
def highlight_best(row):
    best = row["mean"]
    file_mask = summary_df["file"] == row["file"]
    group_min = summary_df.loc[file_mask, "mean"].min()

    if best == group_min:
        return ["background-color: lightgreen"] * len(row)
    else:
        return [""] * len(row)


styled_summary = (
    summary_df
    .style
    .apply(highlight_best, axis=1)
    .format({
        "mean": "{:.4f}",
        "half_ci": "{:.4f}",
        "rel_error_%": "{:.2f}"
    })
)

styled_summary


,file,method,mean,std,n,half_ci,rel_error_%
0,BELO HORIZONTE.csv,doy,121.9797,11.039301,39,3.4647,2.84
1,BELO HORIZONTE.csv,interpolation,141.8146,10.277362,39,3.2256,2.27
2,BELO HORIZONTE.csv,smoothed_log,145.4373,13.815523,39,4.3360,2.98
3,BELO HORIZONTE.csv,smoothed_original,115.9497,10.551021,39,3.3115,2.86
4,CRUZEIRO DO SUL (ACRE).csv,doy,139.8571,9.936142,30,3.5556,2.54
5,CRUZEIRO DO SUL (ACRE).csv,interpolation,227.3846,11.897439,30,4.2574,1.87
6,CRUZEIRO DO SUL (ACRE).csv,smoothed_log,184.9248,13.084903,30,4.6824,2.53
7,CRUZEIRO DO SUL (ACRE).csv,smoothed_original,148.1191,11.181875,30,4.0014,2.70
8,GARANHUNS (PERNAMBUCO).csv,doy,45.7068,6.313366,90,1.3044,2.85
9,GARANHUNS (PERNAMBUCO).csv,interpolation,51.1639,6.416176,90,1.3256,2.59


In [75]:
# ======================================================
# LATEX EXPORT (FINAL CLEAN VERSION)
# ======================================================

import numpy as np

# ======================================================
# 1) SUMMARY TABLE (mean ± CI, bold best)
# ======================================================

summary_latex = summary_df.copy()

# ---- create mean ± CI column ----
summary_latex["mean_ci"] = (
    summary_latex["mean"].round(4).astype(str)
    + " $\\pm$ "
    + summary_latex["half_ci"].round(4).astype(str)
)

# ---- bold best (lowest MSE) within each file ----
mins = summary_latex.groupby("file")["mean"].transform("min")

summary_latex["mean_ci"] = np.where(
    summary_latex["mean"] == mins,
    "\\textbf{" + summary_latex["mean_ci"] + "}",
    summary_latex["mean_ci"]
)

# ---- select + rename columns (safe & explicit) ----
summary_latex = summary_latex[[
    "file",
    "method",
    "mean_ci",
    "rel_error_%"
]]

summary_latex.columns = [
    "File",
    "Method",
    "MSE ($\\pm$ CI)",
    "Rel. Error (\\%)"
]

# ---- export ----
summary_latex.to_latex(
    "mse_summary.tex",
    index=False,
    escape=False,            # IMPORTANT for bold + math
    column_format="llcc",
    float_format="%.4f"
)


# ======================================================
# 2) PAIRED MEANS TEST TABLE
# ======================================================

tests_latex = tests_df.copy()

# ---- significance column ----
tests_latex["signif"] = np.where(
    tests_latex["p_value"] < 0.05,
    "\\textbf{Yes}",
    "No"
)

# ---- keep ONLY desired columns (avoid mismatch bugs) ----
tests_latex = tests_latex[[
    "file",
    "method_A",
    "method_B",
    "mean_A",
    "mean_B",
    "diff(A-B)",
    "p_value",
    "signif"
]]

# ---- rounding ----
tests_latex = tests_latex.round(4)

# ---- rename ----
tests_latex.columns = [
    "File",
    "Method A",
    "Method B",
    "Mean A",
    "Mean B",
    "Diff",
    "$p$-value",
    "Significant"
]

# ---- export ----
tests_latex.to_latex(
    "mse_tests.tex",
    index=False,
    escape=False,
    column_format="lllrrrrc"
)


print("\nLaTeX files successfully created:")
print("  -> mse_summary.tex")
print("  -> mse_tests.tex")



LaTeX files successfully created:
  -> mse_summary.tex
  -> mse_tests.tex
